In [ ]:
import torch
import shap
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from bertviz import head_view, model_view

# Load your fine-tuned model and tokenizer
# Replace with the path to your actual checkpoint: e.g., "../checkpoints/bart_satire_final"
MODEL_PATH = "facebook/bart-base" 
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# output_attentions=True is strictly required for BertViz to capture the attention matrices
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, output_attentions=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model and tokenizer loaded on {device}")

In [ ]:
def f(x):
    # Wrapper function for SHAP text masking
    inputs = tokenizer(x, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=50)
    return [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]

# Initialize the SHAP Explainer
explainer = shap.Explainer(f, tokenizer)

# Example factual headline
factual_text = ["Local man forgets to buy milk at the grocery store."]

# Calculate and plot SHAP values
shap_values = explainer(factual_text)
shap.plots.text(shap_values)

In [ ]:
def visualize_attention(encoder_text, decoder_text):
    """
    Generates an interactive BertViz visualization of the model's attention.
    For style transfer, cross-attention (how the satirical output attends to the factual input)
    is the most critical mechanism to observe.
    """
    # 1. Tokenize the input (factual headline)
    encoder_inputs = tokenizer(encoder_text, return_tensors="pt", add_special_tokens=True).to(device)
    encoder_input_ids = encoder_inputs.input_ids
    
    # 2. Tokenize the target (satirical headline)
    # Using the target_tokenizer context ensures the sequence is formatted correctly for the decoder
    with tokenizer.as_target_tokenizer():
        decoder_inputs = tokenizer(decoder_text, return_tensors="pt", add_special_tokens=True).to(device)
        decoder_input_ids = decoder_inputs.input_ids
        
    # 3. Forward pass to extract attention matrices
    with torch.no_grad():
        outputs = model(input_ids=encoder_input_ids, decoder_input_ids=decoder_input_ids)
        
    # 4. Convert IDs back to string tokens for the visualization labels
    encoder_tokens = tokenizer.convert_ids_to_tokens(encoder_input_ids[0])
    decoder_tokens = tokenizer.convert_ids_to_tokens(decoder_input_ids[0])
    
    # 5. Render the BertViz Head View
    # Note: In a Jupyter Notebook, this will render an interactive HTML widget.
    print("Select 'Cross' to view how the Decoder (Satire) attends to the Encoder (Fact).")
    
    # You can swap head_view with model_view for a macro-level perspective
    head_view(
        encoder_attention=outputs.encoder_attentions,
        decoder_attention=outputs.decoder_attentions,
        cross_attention=outputs.cross_attentions,
        encoder_tokens=encoder_tokens,
        decoder_tokens=decoder_tokens
    )

# Define the source text and the text the model generated (or a ground truth target)
factual_headline = "Local man forgets to buy milk at the grocery store."
satirical_headline = "Local man bravely embarks on perilous grocery quest, returns empty-handed."

# Generate the interactive visualization
visualize_attention(factual_headline, satirical_headline)